# Project: Forecasting Approach Evaluation and Recommendation

The board meeting is in two weeks. Your VP of Sales presented a revenue target of **$4.1M per month by December 2026** — 18% year-over-year growth. The number came from pasting a data export into a generative AI tool and asking it to forecast.

Your data engineering team has already audited the VP's original query and found several bugs that inflated recent revenue figures and made the growth trend look steeper than it actually is. They've corrected the issues and handed you a **clean dataset** (`revenue.csv`) with 60 months of monthly revenue from January 2021 through December 2025.

**Your job:** Take the clean data and figure out what the numbers actually say. Build progressively sophisticated forecasts — from simple baselines through classical ARIMA/SARIMAX to neural models and foundation models — and deliver a stakeholder-ready recommendation with prediction intervals that a non-technical executive can act on.

---

## Deliverables

1. This completed notebook with all phases implemented
2. A comparison table showing all models, their December 2026 forecasts, prediction intervals, and accuracy metrics
3. At least three charts: baseline comparison, classical model forecasts with intervals, modern model forecasts
4. A written recommendation (500–1000 words) answering:
   - What is the realistic range for December 2026 revenue?
   - What should the company plan for instead of $4.1M?
   - Which forecasting approach would you recommend for ongoing use, and why?

## Setup

In [ ]:
!pip install scipy statsmodels darts pytorch_lightning

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from darts import TimeSeries
from darts.models import NBEATSModel, ARIMA as DartsARIMA
from darts.metrics import mae, rmse, mape
from darts.utils.likelihood_models import QuantileRegression
from scipy import stats
from pathlib import Path

VP_TARGET = 4_100_000
HORIZON = 12  # months ahead (Jan 2026 through Dec 2026)

## Load Data

The clean revenue dataset has been prepared by your data engineering team. It contains monthly revenue from January 2021 through December 2025.

In [ ]:
df = pd.read_csv(
    "../data/revenue.csv",
    parse_dates=["date"],
    index_col="date",
)
df.index.freq = "MS"
revenue = df["revenue"]

print(f"Date range: {revenue.index[0].date()} to {revenue.index[-1].date()}")
print(f"Number of months: {len(revenue)}")
print(f"Revenue range: ${revenue.min()/1e6:.2f}M to ${revenue.max()/1e6:.2f}M")

revenue.plot(figsize=(14, 5), color="black", title="Monthly Revenue")
plt.ylabel("Revenue ($)")
plt.tight_layout()
plt.show()

## Helper Functions

Use these helper functions for the baseline forecasts in Phase 1.

In [ ]:
def naive_forecast(series, horizon):
    """Repeat the last observed value for `horizon` months."""
    last = series.iloc[-1]
    future = pd.date_range(
        series.index[-1] + pd.DateOffset(months=1), periods=horizon, freq="MS"
    )
    return pd.Series(last, index=future)


def moving_average_forecast(series, window, horizon):
    """Repeat the trailing `window`-month average for `horizon` months."""
    avg = series.iloc[-window:].mean()
    future = pd.date_range(
        series.index[-1] + pd.DateOffset(months=1), periods=horizon, freq="MS"
    )
    return pd.Series(avg, index=future)


def trend_forecast(series, horizon):
    """Fit a linear trend and extrapolate for `horizon` months."""
    t = np.arange(len(series))
    coeffs = np.polyfit(t, series.values, 1)
    future_t = np.arange(len(series), len(series) + horizon)
    future = pd.date_range(
        series.index[-1] + pd.DateOffset(months=1), periods=horizon, freq="MS"
    )
    return pd.Series(np.polyval(coeffs, future_t), index=future)


def plot_forecasts(actuals, forecasts, target=VP_TARGET, title="Forecast Comparison"):
    """Plot actuals and multiple forecast series against the VP target."""
    fig, ax = plt.subplots(figsize=(14, 6))
    ax.plot(actuals.index, actuals / 1e6, color="black", linewidth=1.5, label="Actual")
    colors = ["tab:blue", "tab:orange", "tab:green", "tab:purple", "tab:red"]
    for i, (name, fc) in enumerate(forecasts.items()):
        ax.plot(fc.index, fc / 1e6, "--", color=colors[i % len(colors)], label=name)
    ax.axhline(
        y=target / 1e6, color="red", linestyle=":", linewidth=2,
        label=f"VP target: ${target/1e6:.1f}M",
    )
    ax.set_ylabel("Revenue ($M)")
    ax.set_title(title)
    ax.legend(loc="upper left")
    plt.tight_layout()
    plt.show()
    return fig

---

## Phase 1: Baseline Forecasts

Before reaching for sophisticated models, establish baselines. Compute simple forecasts over a 12-month horizon and compare each to the VP's $4.1M target.

**TODO:**
1. Create naive forecast (repeat last value)
2. Create 6-month and 12-month moving average forecasts
3. Create linear trend forecast
4. Compare each December 2026 forecast to the VP target
5. Use `plot_forecasts()` to visualize all forecasts

In [ ]:
# PHASE 1 TODO: Create baseline forecasts
#
# HINTS:
# - Use naive_forecast(revenue, HORIZON) for the naive forecast
# - Use moving_average_forecast(revenue, window, HORIZON) for MA forecasts
# - Use trend_forecast(revenue, HORIZON) for the linear trend
# - Store forecasts in a dict: {'Naive': naive_fc, '6MA': ma6_fc, ...}
# - Access December 2026 value using .iloc[-1] or the specific date
# - Compare each forecast to VP_TARGET (4.1M)

# Your code here:




# Print December 2026 forecasts vs VP target
# Example: print(f"Naive: ${naive_fc.iloc[-1]/1e6:.2f}M (gap: ${(naive_fc.iloc[-1] - VP_TARGET)/1e6:.2f}M)")

In [ ]:
# PHASE 1 TODO: Visualize baseline forecasts
#
# Use plot_forecasts(revenue, forecasts_dict, VP_TARGET, title="Phase 1: Baseline Forecasts")

# Your code here:


---

## Phase 2: Classical Models

Fit ARIMA and SARIMAX models to generate forecasts with prediction intervals. Run residual diagnostics (AIC, Ljung-Box, normality tests).

### Phase 2, Step 1: Train/Test Split

Split the data at December 2024. Train on data through December 2024, test on January 2025 through December 2025.

In [ ]:
# PHASE 2 TODO: Create train/test split
#
# Split at December 2024:
# - train = revenue through 2024-12
# - test = revenue from 2025-01 through 2025-12

# Your code here:



# print(f"Train: {train.index[0]} to {train.index[-1]} ({len(train)} months)")
# print(f"Test:  {test.index[0]} to {test.index[-1]} ({len(test)} months)")

### Phase 2, Step 2: Fit ARIMA Model

Fit an ARIMA(2,1,1) model with a constant trend.

**HINT:** Use `SARIMAX(train, order=(2,1,1), trend='c').fit(disp=False)`

In [ ]:
# PHASE 2 TODO: Fit ARIMA(2,1,1) with trend='c'
#
# HINT: arima = SARIMAX(train, order=(2,1,1), trend='c').fit(disp=False)

# Your code here:



# Print model summary
# print(arima.summary())

### Phase 2, Step 3: Fit SARIMAX Model

Fit a SARIMAX(1,1,1)(1,1,1,12) model with a constant trend to capture seasonality.

**HINT:** Use `SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12), trend='c').fit(disp=False)`

In [ ]:
# PHASE 2 TODO: Fit SARIMAX(1,1,1)(1,1,1,12) with trend='c'
#
# HINT: sarimax = SARIMAX(train, order=(1,1,1), seasonal_order=(1,1,1,12), trend='c').fit(disp=False)

# Your code here:



# Print model summary
# print(sarimax.summary())

### Phase 2, Step 4: Run Diagnostics

Calculate AIC, run Ljung-Box test, and check normality of residuals for both models.

In [ ]:
# PHASE 2 TODO: Run diagnostics on ARIMA model
#
# 1. Get AIC: arima.aic
# 2. Run Ljung-Box test: acorr_ljungbox(arima.resid, lags=10)
# 3. Test normality: stats.jarque_bera(arima.resid)

# Your code here:



# print(f"\nARIMA AIC: {arima_aic:.2f}")
# print(f"Ljung-Box p-value: {lb_pvalue:.4f} (want > 0.05 for white noise)")
# print(f"Jarque-Bera p-value: {jb_pvalue:.4f} (want > 0.05 for normality)")

In [ ]:
# PHASE 2 TODO: Run diagnostics on SARIMAX model
#
# Same diagnostics as above but for the SARIMAX model

# Your code here:



# print(f"\nSARIMAX AIC: {sarimax_aic:.2f}")
# print(f"Ljung-Box p-value: {lb_pvalue:.4f}")
# print(f"Jarque-Bera p-value: {jb_pvalue:.4f}")

### Phase 2, Step 5: Evaluate on Test Set

Generate forecasts for the test period (Jan 2025 - Dec 2025) and calculate MAE and RMSE.

In [ ]:
# PHASE 2 TODO: Generate forecasts and evaluate on test set
#
# HINTS:
# - arima_forecast = arima.get_forecast(steps=len(test))
# - arima_mean = arima_forecast.predicted_mean
# - mae = np.mean(np.abs(test - arima_mean))
# - rmse = np.sqrt(np.mean((test - arima_mean)**2))

# Your code here:



# print(f"ARIMA MAE: ${arima_mae:,.0f}, RMSE: ${arima_rmse:,.0f}")
# print(f"SARIMAX MAE: ${sarimax_mae:,.0f}, RMSE: ${sarimax_rmse:,.0f}")

### Phase 2, Step 6: Refit on Full Data and Forecast to December 2026

Now refit both models on the full dataset and generate 12-month forecasts with prediction intervals.

In [ ]:
# PHASE 2 TODO: Refit on full data and generate 12-month forecasts
#
# 1. Refit ARIMA on full `revenue` series
# 2. Refit SARIMAX on full `revenue` series
# 3. Generate forecasts: get_forecast(steps=HORIZON)
# 4. Extract predicted_mean and conf_int(alpha=0.05)

# Your code here:



# Create forecast index: pd.date_range('2026-01-01', periods=HORIZON, freq='MS')
# Store December 2026 values for comparison
# dec_2026_arima = arima_mean.iloc[-1]
# dec_2026_sarimax = sarimax_mean.iloc[-1]

### Phase 2, Step 7: Calculate Probability of Reaching $4.1M

Calculate the probability that December 2026 revenue exceeds the VP's $4.1M target.

In [ ]:
# PHASE 2 TODO: Calculate probability of reaching VP target
#
# FORMULA:
# - se = (ci_upper - mean) / 1.96  (standard error from 95% CI)
# - z = (VP_TARGET - mean) / se    (z-score for target)
# - prob = 1 - stats.norm.cdf(z)   (probability of exceeding target)

# Your code here:



# print(f"Probability of reaching $4.1M:")
# print(f"  ARIMA:   {arima_prob:.1%}")
# print(f"  SARIMAX: {sarimax_prob:.1%}")

In [ ]:
# PHASE 2 TODO: Plot classical model forecasts with prediction intervals
#
# Create a plot showing:
# - Historical revenue (black)
# - ARIMA forecast with confidence interval (blue)
# - SARIMAX forecast with confidence interval (orange)
# - VP target line (red dashed)

# Your code here:


---

## Phase 3: Modern Models

Train a neural forecasting model (N-BEATS) and run a foundation model zero-shot forecast (Chronos-2) using Darts.

### Phase 3, Step 1: Convert to Darts TimeSeries

In [ ]:
# PHASE 3 TODO: Convert to Darts TimeSeries
#
# HINT: ts = TimeSeries.from_dataframe(df.reset_index(), time_col='date', value_cols='revenue', freq='MS')

# Your code here:



# Split for training (same Dec 2024 split)
# train_ts, test_ts = ts.split_before(pd.Timestamp('2025-01-01'))

### Phase 3, Step 2: Train N-BEATS Model

Train an N-BEATS model with quantile regression for prediction intervals.

In [ ]:
# PHASE 3 TODO: Train N-BEATS model
#
# HINTS:
# - model = NBEATSModel(input_chunk_size=12, output_chunk_size=HORIZON, ...)
# - model.fit(train_ts, epochs=50, verbose=True)
# - forecast = model.predict(HORIZON)

# Your code here:



# Get December 2026 forecast value
# nbeats_dec_2026 = forecast.values()[-1][0]

### Phase 3, Step 3: Evaluate N-BEATS and Run Chronos-2

Evaluate N-BEATS on the test set, then run Chronos-2 zero-shot forecast.

In [ ]:
# PHASE 3 TODO: Evaluate N-BEATS on test set
#
# HINTS:
# - pred = model.predict(len(test_ts))
# - mae(test_ts, pred), rmse(test_ts, pred)

# Your code here:



# print(f"N-BEATS MAE: ${nbeats_mae:,.0f}, RMSE: ${nbeats_rmse:,.0f}")

In [ ]:
# PHASE 3 TODO: Run Chronos-2 or fallback to ARIMA in Darts
#
# Try to use Chronos-2 if available, otherwise use Darts ARIMA
#
# HINT for Darts ARIMA:
# - arima_darts = DartsARIMA()
# - arima_darts.fit(train_ts)
# - forecast = arima_darts.predict(HORIZON)

# Your code here:



# Store December 2026 forecast
# chronos_dec_2026 = forecast.values()[-1][0]

In [ ]:
# PHASE 3 TODO: Plot modern model forecasts
#
# Plot N-BEATS and Chronos-2 (or fallback) forecasts against historical data

# Your code here:


---

## Phase 4: Comparison and Recommendation

Bring everything together with a comparison table and stakeholder summary.

### Phase 4, Step 1: Build Comparison Table

In [ ]:
# PHASE 4 TODO: Build comparison table
#
# Create a DataFrame with columns:
# - Model: name of the model
# - Dec 2026 Forecast: predicted value
# - 95% CI Lower: lower bound
# - 95% CI Upper: upper bound
# - MAE: mean absolute error on test set
# - RMSE: root mean squared error on test set
# - P(Reach $4.1M): probability of hitting target

# Your code here:



# Display the table
# print(comparison_df.to_string(index=False))

### Phase 4, Step 2: Three-Response Analysis

Analyze the results using the three-response framework:

**Convergence:** Do the models agree on the likely range?
- Compare the central forecasts across all models
- Do they cluster around a similar value or diverge widely?
- What does agreement/disagreement tell us about forecast reliability?

**Divergence:** Where do the models disagree and why?
- Which models produce higher/lower forecasts?
- How do the prediction intervals differ in width?
- What assumptions might explain the differences (e.g., trend continuation vs. mean reversion)?

**Confidence:** How should we interpret the uncertainty?
- What is the realistic range for December 2026 revenue?
- How much confidence should stakeholders have in these projections?
- What factors could shift the actual outcome outside these ranges?

In [ ]:
# PHASE 4 TODO: Three-Response Analysis
#
# Write your analysis here addressing:
#
# CONVERGENCE:
# - Calculate the range of Dec 2026 forecasts across all models
# - Calculate the average forecast
# - Comment on whether models agree

# Your analysis here:


# DIVERGENCE:
# - Identify which models are optimistic vs pessimistic
# - Compare prediction interval widths
# - Explain likely reasons for differences

# Your analysis here:


# CONFIDENCE:
# - Calculate overall realistic range (min lower CI to max upper CI)
# - Calculate probability-weighted expectation
# - Identify key risks that could change the outcome

# Your analysis here:


### Phase 4, Step 3: Written Recommendation

Write a 500–1000 word stakeholder-ready recommendation addressing:

1. **What is the realistic range for December 2026 revenue?**
   - Based on model forecasts and prediction intervals
   - Consider the spread across different approaches

2. **What should the company plan for instead of $4.1M?**
   - Provide a specific target or range
   - Explain the gap between models and the VP's original target
   - Discuss probability of achieving different revenue levels

3. **Which forecasting approach would you recommend for ongoing use, and why?**
   - Compare accuracy, interpretability, and practical considerations
   - Consider maintenance requirements and stakeholder understanding
   - Recommend a specific approach with justification

**REPLACE THIS WITH YOUR WRITTEN RECOMMENDATION (500–1000 words):**

[Your recommendation here...]